# 14 — Evaluation Benchmark (Milestone M6)

**DSML stage:** evaluation. A 20-question gold benchmark over the live 13-filer graph, scoring the
**entity-first hybrid** pipeline against a **vector-only baseline** — the number that justifies the graph.

Metrics (definitions per the feasibility studies; implemented as transparent LLM-judge + programmatic
checks with our hardened `llm_json` — not the RAGAS framework, which needs LangChain adapters we don't use):

| Metric | How |
|---|---|
| **Answer Faithfulness** | Sonnet judge: claims in the answer supported by retrieved context (ratio) |
| **Context Precision** | Haiku judge: fraction of retrieved chunks relevant to the question |
| **Correctness** | programmatic (exact numbers from XBRL, expected-entity substrings, refusal detection) + judge for open questions |
| **Citation validity** | programmatic: every cited chunk id must exist in the retrieved set |
| **Numeric consistency** | programmatic: financial figures must match XBRL exactly |

**Every expected answer below was verified against the live graph/XBRL before being written down.**
Estimated cost: ~40 answers + ~80 judge calls ≈ **$2–3**. Target: faithfulness ≥ 0.85 (docs), hard floor 0.75.

## A note on methodology — why not RAGAS (for now)?

**The metrics below are the industry-standard RAG evaluation metrics** — *Faithfulness/Groundedness,
Context Precision, Answer Correctness* — the same vocabulary popularized by RAGAS and shared by TruLens
("RAG triad"), DeepEval, and Arize Phoenix. What we skip is the *framework*, not the standard. Under the
hood, RAGAS does exactly what this notebook does: send judge prompts to an LLM and compute ratios. Here,
those prompts are visible, debuggable cells instead of calls hidden behind an API.

**Why the framework doesn't fit at this stage:**

1. **Integration cost** — RAGAS 0.4 requires an LLM backend via LangChain-style adapters (new
   dependencies, untested glue). This project's track record is clear: every unverified integration has
   cost a failed run. For a 20-question benchmark, a transparent judge built on our already-hardened
   `llm_json` helper (Sonnet 5 quirks, truncation, transient-error backoff) is the lower-risk path.

2. **RAGAS is generic text-QA evaluation; our system's claims are stronger than generic.** The metrics
   that make a *financial knowledge-graph* system credible are domain-specific, and mostly don't need an
   LLM judge at all:
   - **Citation validity** — every cited chunk id must exist in the retrieved set (programmatic, exact)
   - **Numeric consistency** — financial figures must match XBRL to the dollar (programmatic, exact)
   - **Temporal consistency** — a risk deleted from the latest 10-K must not be presented as current
   - **Refusal correctness** — questions whose facts are deliberately absent from the corpus
     (Samsung revenue, earnings calls) must be declined, not hallucinated

   RAGAS covers none of these; we would hand-build them regardless. And where ground truth is
   deterministic, exact programmatic checks beat any LLM judge — cheaper, reproducible, bias-free.

3. **Ground truths here are verified, not assumed** — every expected value/entity in the benchmark was
   checked against the live graph and XBRL before being written down.

**Known limitation (applies to RAGAS equally):** the faithfulness judge is Claude Sonnet evaluating
Claude Sonnet's answers — a self-preference bias risk. Mitigation planned for the SDK phase (M7):
a second scorer (RAGAS or DeepEval) over the shared metrics as cross-validation, a different judge
model family for sensitive comparisons, and RAGAS's synthetic testset generator to grow this benchmark
from 20 toward the feasibility studies' 50 questions.

In [1]:
import json
import os
import re
import time as _time
from pathlib import Path

import litellm
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from litellm import completion
from neo4j import GraphDatabase
from pydantic import BaseModel, ValidationError
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]))
driver.verify_connectivity()
LLM_MODEL = os.environ["LLM_MODEL"]
JUDGE_MODEL = os.environ["LLM_MODEL"]          # faithfulness needs judgment -> Sonnet
CHEAP_JUDGE = "anthropic/claude-haiku-4-5"       # per-chunk relevance -> Haiku
CANONICAL = json.loads((PROJECT_ROOT / "artifacts/canonical_entities.json").read_text())

def load_embedder(name):
    try: return SentenceTransformer(name, local_files_only=True)
    except OSError: return SentenceTransformer(name)
model = load_embedder(os.getenv("EMBEDDING_MODEL", "Qwen/Qwen3-Embedding-0.6B"))

def run_cypher(q, **params):
    with driver.session() as s:
        return [dict(r) for r in s.run(q, **params)]

def embed_query(q):
    if "query" in (model.prompts or {}):
        return model.encode([q], prompt_name="query", normalize_embeddings=True)[0].tolist()
    return model.encode([q], normalize_embeddings=True)[0].tolist()

TRANSIENT = (litellm.APIConnectionError, litellm.ServiceUnavailableError,
             litellm.InternalServerError, litellm.RateLimitError, litellm.Timeout)

def llm_json(prompt, model_cls, model_name=None, max_tokens=2000, thinking_off=True):
    """The hardened structured-call helper (same failure handling as notebook 12)."""
    model_name = model_name or LLM_MODEL
    kwargs = {"thinking": {"type": "disabled"}} if thinking_off else {}
    messages, budget, last_err = [{"role": "user", "content": prompt}], max_tokens, "unknown"
    for attempt in range(4):
        try:
            resp = completion(model=model_name, messages=messages, max_tokens=budget, num_retries=2, **kwargs)
        except TRANSIENT as e:
            wait = [15, 60, 180, 300][attempt]
            print(f"    transient ({type(e).__name__}) — waiting {wait}s"); _time.sleep(wait); continue
        choice = resp.choices[0]
        content = choice.message.content
        if not content:
            messages = [{"role": "user", "content": prompt}]; continue
        if choice.finish_reason == "length":
            budget = min(budget * 2, 8000); messages = [{"role": "user", "content": prompt}]; continue
        raw = re.sub(r"^```(json)?|```$", "", content.strip(), flags=re.MULTILINE).strip()
        try:
            return model_cls.model_validate_json(raw)
        except ValidationError as e:
            last_err = str(e)[:200]
            messages = [{"role": "user", "content": prompt}, {"role": "assistant", "content": raw},
                        {"role": "user", "content": f"Invalid JSON: {e}. Reply with corrected JSON only."}]
    raise RuntimeError(f"llm_json failed — {last_err}")
print("ready")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

ready


## 1. The two systems under test — hybrid (graph+vector) vs vector-only baseline

In [ ]:
ALIAS_RES = [(re.compile(rf"\b{re.escape(a)}\b", re.I), (name, spec["entity_id"]))
             for name, spec in CANONICAL.items() for a in {name, *spec["aliases"]}]

# Vector search uses the SEARCH clause (Neo4j 2026.x; grammar verified live: single-variable MATCH,
# filters AFTER the search, score via `SCORE AS`). v3 additions from the first eval run's findings:
# - hybrid now surfaces the BITEMPORAL layer (dropped risk lineages from notebook 13) — temporal
#   questions scored 0.67/0.00 because retrieval never exposed what the graph knew
# - answer() returns the FULL context so the faithfulness judge sees exactly what the answering
#   model saw (judging hybrid answers against excerpts alone was the metric bug: 0.95 correct
#   yet "0.36 faithful" — the graph-block claims were invisible to the judge)

def hybrid_retrieve(question, k_chunks=8, hops=2):
    anchors = {name: eid for pat, (name, eid) in ALIAS_RES if pat.search(question)}
    anchor_ids = list(anchors.values()) or [1045810]
    edges = run_cypher(
        f"""MATCH (a:Company) WHERE a.cik IN $ids
        MATCH p = (a)-[r:SUPPLIES_TO|DEPENDS_ON|CUSTOMER_OF|COMPETES_WITH|AFFECTED_BY*1..{hops}]-(b)
        WHERE (b:Company OR b:ExportControl)
        UNWIND relationships(p) AS rel
        RETURN DISTINCT coalesce(startNode(rel).name, startNode(rel).title) AS source, type(rel) AS relation,
               coalesce(endNode(rel).name, endNode(rel).title) AS target,
               rel.status AS status, rel.evidence_quote AS quote, rel.evidence_chunk_ids AS chunk_ids""",
        ids=anchor_ids)
    metrics = run_cypher(
        """MATCH (c:Company)-[:REPORTS_METRIC]->(m:Metric) WHERE c.cik IN $ids
        RETURN c.name AS company, m.metric AS metric, m.value AS value,
               toString(m.period_start) AS period_start, toString(m.period_end) AS period_end
        ORDER BY m.period_end DESC LIMIT 20""", ids=anchor_ids)
    risks = run_cypher(
        """MATCH (rf:RiskFactor)
        SEARCH rf IN (VECTOR INDEX risk_embedding FOR $vec LIMIT 40) SCORE AS score
        MATCH (a:Company)-[d:DISCLOSES_RISK {status:'Active'}]->(rf)-[:HAS_EVIDENCE]->(e:EvidenceSpan)
        WHERE a.cik IN $ids
        RETURN a.name AS company, rf.summary AS summary, rf.category AS category,
               e.chunk_id AS chunk_id, score ORDER BY score DESC LIMIT 6""",
        ids=anchor_ids, vec=embed_query(question))
    temporal = run_cypher(
        """MATCH (a:Company)-[d:DISCLOSES_RISK {status:'Deleted'}]->(rf:RiskFactor)
        WHERE a.cik IN $ids AND rf.lineage_id IS NOT NULL
        WITH a.name AS company, rf.lineage_id AS lineage,
             toString(min(rf.first_seen)) AS first_seen, toString(max(rf.last_seen)) AS last_seen,
             collect(rf.summary)[0] AS example
        RETURN company, lineage, first_seen, last_seen, example
        ORDER BY last_seen DESC LIMIT 10""", ids=anchor_ids)
    chunks = run_cypher(
        """MATCH (node:EvidenceSpan)
        SEARCH node IN (VECTOR INDEX evidence_embedding FOR $vec LIMIT 60) SCORE AS score
        MATCH (node)-[:MENTIONS]->(c:Company) WHERE c.cik IN $ids
        RETURN DISTINCT node.chunk_id AS chunk_id, score, node.text AS text, node.source_url AS source_url
        ORDER BY score DESC LIMIT $k""", ids=anchor_ids, vec=embed_query(question), k=k_chunks)
    return {"anchors": anchors, "edges": edges, "metrics": metrics, "risks": risks,
            "temporal": temporal, "chunks": chunks}

def vector_retrieve(question, k_chunks=8):
    chunks = run_cypher(
        """MATCH (node:EvidenceSpan)
        SEARCH node IN (VECTOR INDEX evidence_embedding FOR $vec LIMIT $k) SCORE AS score
        RETURN node.chunk_id AS chunk_id, score, node.text AS text, node.source_url AS source_url""",
        k=k_chunks, vec=embed_query(question))
    return {"anchors": {}, "edges": [], "metrics": [], "risks": [], "temporal": [], "chunks": chunks}

ANSWER_PROMPT = """You are a semiconductor supply-chain analyst. Answer the question using ONLY the context below,
retrieved from SEC filings via a knowledge graph.

Rules:
- Cite evidence after every factual sentence using [chunk_id] (ids appear in the context).
- KNOWN RELATIONSHIPS, REPORTED METRICS and DROPPED RISK LINEAGES come from the knowledge graph.
- If the context does not contain the answer, say so plainly — never fill gaps from memory.
- Be concise. Use bullet lists for enumerations.

QUESTION: {question}

=== KNOWN RELATIONSHIPS ===
{edges_block}

=== REPORTED METRICS (deterministic, from XBRL) ===
{metrics_block}

=== DISCLOSED RISKS (currently active, semantically ranked) ===
{risks_block}

=== RISK LINEAGES DROPPED FROM THE LATEST ANNUAL REPORT (bitemporal layer) ===
{temporal_block}

=== SOURCE EXCERPTS ===
{chunks_block}
"""

def build_blocks(r):
    valid_ids = set()
    e_lines = []
    for e in r["edges"]:
        ids = e.get("chunk_ids") or []
        valid_ids.update(ids)
        e_lines.append(f"- {e['source']} {e['relation']} {e['target']} (status={e.get('status')}) "
                       f"{' '.join('[' + i + ']' for i in ids[:3])}")
    m_lines = [f"- {m['company']} {m['metric']} for period {m['period_start']}..{m['period_end']}: "
               f"{m['value']:,.0f} USD" for m in r["metrics"]]
    k_lines = []
    for k in r["risks"]:
        valid_ids.add(k["chunk_id"])
        k_lines.append(f"- {k['company']} ({k['category']}): {k['summary']} [{k['chunk_id']}]")
    t_lines = [f"- {t['company']}: disclosed {t['first_seen']} through {t['last_seen']}, then dropped — "
               f"e.g. {t['example'][:120]}" for t in r["temporal"]]
    c_lines = []
    for c in r["chunks"]:
        valid_ids.add(c["chunk_id"])
        c_lines.append(f"[{c['chunk_id']}]\n{c['text']}\n")
    blocks = ("\n".join(e_lines) or "(none)", "\n".join(m_lines) or "(none)",
              "\n".join(k_lines) or "(none)", "\n".join(t_lines) or "(none)",
              "\n".join(c_lines) or "(none)")
    full_context = ("RELATIONSHIPS:\n{}\n\nMETRICS:\n{}\n\nACTIVE RISKS:\n{}\n\n"
                    "DROPPED RISK LINEAGES:\n{}\n\nEXCERPTS:\n{}").format(*blocks)
    return blocks, full_context, valid_ids

CITE_RE = re.compile(r"\[([0-9\-]+:[IVX]+\.[0-9A-Z]+:[0-9]{4})\]")

def answer(question, retriever):
    r = retriever(question)
    (e_b, m_b, k_b, t_b, c_b), full_context, valid_ids = build_blocks(r)
    prompt = ANSWER_PROMPT.format(question=question, edges_block=e_b, metrics_block=m_b,
                                  risks_block=k_b, temporal_block=t_b, chunks_block=c_b)
    for attempt in range(4):
        try:
            resp = completion(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}],
                              max_tokens=1200, thinking={"type": "disabled"}, num_retries=2)
            break
        except TRANSIENT as e:
            _time.sleep([15, 60, 180, 300][attempt])
    text = resp.choices[0].message.content or ""
    cited = set(CITE_RE.findall(text))
    return {"answer": text, "cited": cited, "valid_ids": valid_ids,
            "hallucinated": cited - valid_ids, "context": full_context, "retrieval": r}
print("systems ready")

## 2. The gold benchmark — 20 questions, every expectation verified against the live graph

Types follow the feasibility studies' taxonomy. `expect` drives programmatic scoring; open questions
carry `judge_notes` for the LLM judge; `refusal` questions must be declined (their facts are NOT in the corpus).

In [3]:
BENCHMARK = [
    # --- numeric (XBRL ground truth, values verified in the graph) ---
    {"id": "N1", "type": "numeric", "q": "What was Nvidia's total revenue for the fiscal year ended January 28, 2024?",
     "expect": {"value": 60_922_000_000}},
    {"id": "N2", "type": "numeric", "q": "What was Nvidia's total revenue for the fiscal year ended January 25, 2026?",
     "expect": {"value": 215_938_000_000}},
    {"id": "N3", "type": "numeric", "q": "What was Nvidia's total revenue for the fiscal year ended January 26, 2025?",
     "expect": {"value": 130_497_000_000}},
    {"id": "N4", "type": "numeric", "q": "What was Microsoft's total revenue for the fiscal year ended June 30, 2025?",
     "expect": {"value": 281_724_000_000}},
    # --- supply-chain dependency (graph edges verified) ---
    {"id": "D1", "type": "dependency", "q": "Which companies does Nvidia depend on for chip manufacturing and assembly?",
     "expect": {"any_of": ["TSMC", "Taiwan Semiconductor", "Samsung", "Foxconn"]}},
    {"id": "D2", "type": "dependency", "q": "Which memory suppliers does Nvidia buy HBM or memory from?",
     "expect": {"any_of": ["Micron", "SK Hynix", "Samsung"]}},
    {"id": "D3", "type": "dependency", "q": "Which foundries does AMD rely on to manufacture its chips?",
     "expect": {"any_of": ["TSMC", "Taiwan Semiconductor", "GlobalFoundries", "Samsung"]}},
    {"id": "D4", "type": "dependency", "q": "Which hyperscaler is disclosed as a customer of AMD in the knowledge graph?",
     "expect": {"any_of": ["Meta"]}},
    # --- regulatory (AFFECTED_BY edges verified: Nvidia, AMD, Broadcom) ---
    {"id": "R1", "type": "regulatory", "q": "Which BIS export-control rules is AMD affected by?",
     "expect": {"any_of": ["Entity List", "Advanced Computing", "Foreign-Produced Direct Product"]}},
    {"id": "R2", "type": "regulatory", "q": "Which companies in the graph are affected by US export-control rules?",
     "expect": {"any_of": ["Nvidia", "AMD", "Broadcom"]}},
    {"id": "R3", "type": "regulatory", "q": "What does Nvidia disclose about export controls affecting its sales to China?",
     "judge_notes": "Should describe US export controls restricting advanced GPU/AI chip sales to China, citing Nvidia filing chunks."},
    # --- temporal (bitemporal layer from notebook 13; 73 deleted NVDA lineages verified) ---
    {"id": "T1", "type": "temporal", "q": "Did Nvidia stop disclosing any risk factors in its latest 10-K that appeared in earlier 10-Ks?",
     "judge_notes": "Correct answer affirms YES — dozens of risk lineages were closed/dropped — ideally with examples."},
    {"id": "T2", "type": "temporal", "q": "What new supply-chain or geopolitical risks appeared in Meta's most recent annual report?",
     "judge_notes": "Should list plausible Meta risks with citations from Meta chunks; 'newly appeared' framing acknowledged."},
    {"id": "T3", "type": "temporal", "q": "How has Nvidia's disclosed risk profile evolved across its recent annual reports?",
     "judge_notes": "Should discuss evolution (e.g., growing AI/export-control emphasis) grounded in cited Nvidia chunks from multiple years."},
    # --- qualitative risk (graph contents verified: TSM 217 risks, ASML 333) ---
    {"id": "Q1", "type": "risk", "q": "What supply-chain related risks does TSMC disclose in its annual report?",
     "judge_notes": "Should present TSMC-disclosed risks with citations from TSM 20-F chunks."},
    {"id": "Q2", "type": "risk", "q": "What geopolitical risks does ASML disclose?",
     "judge_notes": "Should present ASML-disclosed risks (export restrictions, China exposure) citing ASML chunks."},
    {"id": "Q3", "type": "risk", "q": "Compare the competition risks disclosed by AMD and Nvidia.",
     "judge_notes": "Should contrast both companies' competition risks, citing chunks from BOTH filers."},
    # --- metric reasoning ---
    {"id": "M1", "type": "numeric", "q": "By how much did Nvidia's annual revenue grow from the fiscal year ended January 28, 2024 to the fiscal year ended January 25, 2026?",
     "expect": {"any_of": ["155", "3.5", "254%", "215.9", "60.9"]},
     "judge_notes": "Correct if consistent with 60.922B -> 215.938B (about +155B, ~3.5x)."},
    # --- refusal (facts NOT in corpus; correct behavior = decline) ---
    {"id": "U1", "type": "refusal", "q": "What is Samsung's total annual revenue according to its SEC filings?"},
    {"id": "U2", "type": "refusal", "q": "What did Nvidia's CEO say on the most recent earnings call?"},
]

bench_path = PROJECT_ROOT / "artifacts" / "benchmark.json"
bench_path.write_text(json.dumps(BENCHMARK, indent=2), encoding="utf-8")
print(f"{len(BENCHMARK)} questions -> {bench_path.relative_to(PROJECT_ROOT)}")
print(f"cost preview: ~{len(BENCHMARK) * 2} answers + ~{len(BENCHMARK) * 4} judge calls ≈ $2-3")

20 questions -> artifacts\benchmark.json
cost preview: ~40 answers + ~80 judge calls ≈ $2-3


## 3. Run both systems over the benchmark (checkpointed per question/system)

In [ ]:
RESULTS_PATH = PROJECT_ROOT / "data" / "processed" / "eval_runs.jsonl"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
done = set()
if RESULTS_PATH.exists():
    done = {(json.loads(l)["id"], json.loads(l)["system"]) for l in RESULTS_PATH.open(encoding="utf-8") if l.strip()}
print(f"{len(done)} runs checkpointed — resuming")

with RESULTS_PATH.open("a", encoding="utf-8") as sink:
    for q in BENCHMARK:
        for system, retriever in (("hybrid", hybrid_retrieve), ("vector", vector_retrieve)):
            if (q["id"], system) in done:
                continue
            a = answer(q["q"], retriever)
            sink.write(json.dumps({"id": q["id"], "system": system, "type": q["type"], "q": q["q"],
                                   "answer": a["answer"], "cited": sorted(a["cited"]),
                                   "valid_ids": sorted(a["valid_ids"]),
                                   "hallucinated": sorted(a["hallucinated"]),
                                   "context": a["context"],
                                   "chunk_texts": {c["chunk_id"]: c["text"] for c in a["retrieval"]["chunks"]}}) + "\n")
            sink.flush()
            print(f"  {q['id']}/{system} done")
runs = [json.loads(l) for l in RESULTS_PATH.open(encoding="utf-8") if l.strip()]
print(f"{len(runs)} runs total")

## 4. Score — programmatic checks + LLM judges

In [ ]:
class Faithfulness(BaseModel):
    total_claims: int
    supported_claims: int
class Relevance(BaseModel):
    verdicts: list[bool]
class Correct(BaseModel):
    correct: bool
    reason: str

FAITH_PROMPT = """Fact-check this answer against ONLY the provided context (graph facts + excerpts).
Count the factual claims in the answer, then count how many are directly supported by the context.
Statements of absence ('the context does not contain X') count as supported when nothing contradicts them.

QUESTION: {q}\n\nANSWER:\n{a}\n\nCONTEXT:\n{ctx}\n\nReply ONLY with JSON: {{"total_claims": N, "supported_claims": K}}"""

REL_PROMPT = """For each numbered excerpt, answer true if it is relevant to answering the question, else false.\n
QUESTION: {q}\n\nEXCERPTS:\n{chunks}\n\nReply ONLY with JSON: {{"verdicts": [true/false, ...]}} in order."""

JUDGE_PROMPT = """Judge whether the answer is a correct response to the question, given the grading notes.\n
QUESTION: {q}\nGRADING NOTES: {notes}\n\nANSWER:\n{a}\n\nReply ONLY with JSON: {{"correct": true/false, "reason": "..."}}"""

REFUSAL_PAT = re.compile(r"does not (contain|include|provide)|not available|no (information|data|filings)|"
                          r"cannot (be )?(determin|answer|find)|isn't|is not in the (context|filings|corpus)|not an SEC filer", re.I)
NUM_PAT = re.compile(r"\$?([0-9][0-9,\.]*)\s*(billion|bn|b\b|million|mn|m\b|trillion)?", re.I)

def parse_numbers(text):
    out = []
    for m in NUM_PAT.finditer(text.replace(",", "")):
        try: v = float(m.group(1).replace(",", ""))
        except ValueError: continue
        unit = (m.group(2) or "").lower()
        mult = {"billion": 1e9, "bn": 1e9, "b": 1e9, "million": 1e6, "mn": 1e6, "m": 1e6, "trillion": 1e12}.get(unit, 1)
        out.append(v * mult)
    return out

bench_by_id = {b["id"]: b for b in BENCHMARK}
scored = []
for run in runs:
    b = bench_by_id[run["id"]]
    row = {"id": run["id"], "system": run["system"], "type": run["type"]}
    row["citation_ok"] = len(run["hallucinated"]) == 0
    row["n_citations"] = len(run["cited"])
    ans = run["answer"]
    # correctness
    if run["type"] == "refusal":
        row["correct"] = bool(REFUSAL_PAT.search(ans))
    elif "expect" in b and "value" in b["expect"]:
        target = b["expect"]["value"]
        row["correct"] = any(abs(v - target) / target < 0.005 for v in parse_numbers(ans))
    elif "expect" in b and "any_of" in b["expect"]:
        row["correct"] = any(s.lower() in ans.lower() for s in b["expect"]["any_of"])
    else:
        v = llm_json(JUDGE_PROMPT.format(q=b["q"], notes=b.get("judge_notes", ""), a=ans[:4000]),
                     Correct, model_name=JUDGE_MODEL, max_tokens=300)
        row["correct"] = v.correct
    # faithfulness (skip refusals — nothing to fact-check)
    if run["type"] != "refusal":
        # judge against the FULL context the answering model saw (graph blocks + excerpts) —
        # judging hybrid answers against excerpts alone falsely marks graph-derived claims unsupported
        ctx = run.get("context") or "\n".join(f"[{cid}] {t[:600]}" for cid, t in run["chunk_texts"].items())
        ctx = ctx[:24000]
        f = llm_json(FAITH_PROMPT.format(q=b["q"], a=ans[:4000], ctx=ctx or "(no context retrieved)"),
                     Faithfulness, model_name=JUDGE_MODEL, max_tokens=200)
        row["faithfulness"] = (f.supported_claims / f.total_claims) if f.total_claims else 1.0
    # context precision (Haiku, one call per run)
    if run["chunk_texts"]:
        numbered = "\n".join(f"{i+1}. {t[:350]}" for i, t in enumerate(run["chunk_texts"].values()))
        rel = llm_json(REL_PROMPT.format(q=b["q"], chunks=numbered), Relevance,
                       model_name=CHEAP_JUDGE, max_tokens=200, thinking_off=False)
        if len(rel.verdicts) == len(run["chunk_texts"]):
            row["context_precision"] = sum(rel.verdicts) / len(rel.verdicts)
    scored.append(row)
    print(f"  scored {run['id']}/{run['system']}")
scored_df = pd.DataFrame(scored)
scored_df.to_json(PROJECT_ROOT / "artifacts" / "eval_scores.json", orient="records", indent=2)
print("scoring complete")

## 5. The headline result — hybrid GraphRAG vs vector-only baseline

In [6]:
summary = (scored_df.groupby("system")
           .agg(correct=("correct", "mean"), faithfulness=("faithfulness", "mean"),
                context_precision=("context_precision", "mean"),
                citation_validity=("citation_ok", "mean"), avg_citations=("n_citations", "mean"))
           .round(3))
by_type = (scored_df.pivot_table(index="type", columns="system", values="correct", aggfunc="mean").round(2))
print("=== overall ==="); print(summary.to_string())
print("\n=== correctness by question type ==="); print(by_type.to_string())
report = {"overall": summary.to_dict(), "by_type": by_type.to_dict(),
          "n_questions": len(BENCHMARK), "scored_runs": len(scored_df)}
(PROJECT_ROOT / "artifacts" / "eval_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("\nreport -> artifacts/eval_report.json")

=== overall ===
        correct  faithfulness  context_precision  citation_validity  avg_citations
system                                                                            
hybrid     0.95         0.365              0.294               0.95            5.7
vector     0.80         0.753              0.325               1.00            3.9

=== correctness by question type ===
system      hybrid  vector
type                      
dependency    1.00     1.0
numeric       1.00     0.8
refusal       1.00     1.0
regulatory    1.00     1.0
risk          1.00     1.0
temporal      0.67     0.0

report -> artifacts/eval_report.json


In [7]:
# Failure review — read these before trusting the aggregates (and to grow the canonical dictionary / prompts)
fails = scored_df[~scored_df["correct"].astype(bool)]
runs_by_key = {(r["id"], r["system"]): r for r in runs}
for _, f in fails.iterrows():
    r = runs_by_key[(f["id"], f["system"])]
    print(f"--- {f['id']}/{f['system']} ({f['type']}) ---")
    print("Q:", r["q"][:100])
    print("A:", r["answer"][:280].replace(chr(10), " "), "\n")

--- N4/vector (numeric) ---
Q: What was Microsoft's total revenue for the fiscal year ended June 30, 2025?
A: The provided context does not contain a specific figure for Microsoft's total consolidated revenue for the fiscal year ended June 30, 2025.  What the excerpts do show for that period: - Microsoft Cloud revenue increased 23% to $168.9 billion in fiscal year 2025 compared with fisc 

--- T1/hybrid (temporal) ---
Q: Did Nvidia stop disclosing any risk factors in its latest 10-K that appeared in earlier 10-Ks?
A: Based on the provided context, I cannot fully answer this question with confidence.  **What the context shows:**  - The retrieved excerpts do not include the full text or a complete list of risk factor headings from Nvidia's latest 10-K (fiscal year ended January 25, 2026) alongs 

--- T1/vector (temporal) ---
Q: Did Nvidia stop disclosing any risk factors in its latest 10-K that appeared in earlier 10-Ks?
A: # Answer  The provided context does not contain sufficient infor

In [8]:
# --- M6 assertion cell ---
hyb = scored_df[scored_df["system"] == "hybrid"]
vec = scored_df[scored_df["system"] == "vector"]
hyb_faith = hyb["faithfulness"].mean()
hyb_correct = hyb["correct"].mean()
assert len(hyb) == len(BENCHMARK) and len(vec) == len(BENCHMARK), "incomplete runs"
assert (hyb["citation_ok"].mean()) >= 0.9, "citation hallucinations above 10% — inspect answer prompt"
assert hyb_faith >= 0.75, f"hybrid faithfulness {hyb_faith:.2f} below hard floor 0.75"
assert hyb_correct >= 0.6, f"hybrid correctness {hyb_correct:.2f} below hard floor 0.6"
target_note = "MEETS" if hyb_faith >= 0.85 else "below"
print(f"M6 COMPLETE — hybrid: correctness {hyb_correct:.0%}, faithfulness {hyb_faith:.0%} "
      f"({target_note} the docs' 0.85 target), citation validity {hyb['citation_ok'].mean():.0%}")
print(f"vs vector baseline: correctness {vec['correct'].mean():.0%}, "
      f"faithfulness {vec['faithfulness'].mean():.0%} — the graph's lift is the difference")
driver.close()

AssertionError: hybrid faithfulness 0.36 below hard floor 0.75